In [1]:
import re
import pandas as pd
import glob
import os
import numpy as np

In [ ]:
folder = "../data/raw/normal"

In [7]:
dfs = []
for file in glob.glob(os.path.join(folder, "*.xlsx")):
    df = pd.read_excel(file, skiprows=3)
    dfs.append(df)

merged = pd.concat(dfs, ignore_index=True, join="outer")
merged.to_csv(f"{folder}/normal.csv", index=False)

In [8]:
healthy = pd.read_csv('../data/normal/normal.csv')

In [9]:
meta_cols = ['Регистрационный номер', 'Возраст компании, лет', 'Вид деятельности/отрасль']

for c in meta_cols:
    if c not in merged.columns:
        merged[c] = pd.NA

id_cols = ['№'] + meta_cols

value_cols = [c for c in healthy.columns if re.match(r'\d{4},', str(c))]

df_normal_long = healthy.melt(id_vars=id_cols, value_vars=value_cols,
                      var_name='Показатель_год', value_name='Значение')
df_normal_long[['Год', 'Показатель']] = df_normal_long['Показатель_год'].str.extract(r'(\d{4}),\s*(.*)')
df_normal_long = df_normal_long.drop(columns='Показатель_год')

df_normal_long['Год'] = pd.to_numeric(df_normal_long['Год'], errors='coerce').astype('Int64')

pivot_index = ['Год'] + meta_cols
df_wide_normal = df_normal_long.pivot_table(
    index=pivot_index,
    columns='Показатель',
    values='Значение',
    aggfunc='first'
).reset_index()

df_wide_normal.columns.name = None

In [10]:
df_wide_normal['next_year_liqudation'] = 0

In [13]:
df_wide_normal = df_wide_normal.sort_values(['Регистрационный номер', 'Год'])

In [14]:
df_cleaned = df_wide_normal.dropna(axis=0, subset=df_wide_normal.loc[:,'Коэффициент быстрой ликвидности, %':'Соотношение совокупного долга к капиталу, %'].columns.to_list(), thresh=6)

In [15]:
counts = df_cleaned['Регистрационный номер'].value_counts().sort_values()
df_cleaned = df_cleaned[df_cleaned['Регистрационный номер'].isin(counts[counts > 2].index)]

In [ ]:
year_of_group = df_cleaned.groupby('Регистрационный номер')['Год']
is_same_size = (year_of_group.last() - year_of_group.first() == year_of_group.count()-1)
df_wide_cleaned = df_cleaned[df_cleaned['Регистрационный номер'].isin(is_same_size[is_same_size].index)]

In [17]:
companies_ages = df_wide_cleaned.groupby('Регистрационный номер')['Возраст компании, лет'].first().sort_values()
print(f'Средний возраст: {companies_ages.mean():.1f}, Макс: {companies_ages.max()}, Мин: {companies_ages.min()}, медиана: {companies_ages.median()}')

Средний возраст: 17.6, Макс: 36.0, Мин: 5.0, медиана: 17.0


In [18]:
companies_revenue = df_wide_cleaned.groupby('Регистрационный номер')['Выручка, RUB'].mean().sort_values()
print(f'Средняя выручка: {companies_revenue.mean():,.1f}, Макс: {companies_revenue.max():,.1f}, Мин: {companies_revenue.min():,.1f}, медиана: {companies_revenue.median():,.1f}')

Средняя выручка: 266,356,825.3, Макс: 3,211,346,200.0, Мин: 159,800.0, медиана: 99,122,800.0


In [19]:
for col in ['Оборачиваемость кредиторской задолженности, разы', 'Оборачиваемость дебиторской задолженности, разы']:
    df_wide_cleaned.loc[:, col] = df_wide_cleaned[col].apply(lambda x: np.nan if x <= 0 else x)
    df_wide_cleaned.loc[:, col] = df_wide_cleaned.groupby('Вид деятельности/отрасль')[col].transform(
    lambda x: x.fillna(x.median()))

/var/folders/rl/0ykyj_7s1wx56ylgms0h9bxh0000gn/T/ipykernel_75710/3666379539.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  lambda x: x.fillna(x.median()))
/var/folders/rl/0ykyj_7s1wx56ylgms0h9bxh0000gn/T/ipykernel_75710/3666379539.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  lambda x: x.fillna(x.median()))
/var/folders/rl/0ykyj_7s1wx56ylgms0h9bxh0000gn/T/ipykernel_75710/3666379539.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=Fa

In [20]:
for col in ['Рентабельность продаж, %', 'Соотношение совокупного долга к капиталу, %', 'Выручка, RUB']:
    df_wide_cleaned[col] = df_wide_cleaned.groupby('Регистрационный номер')[col].transform(
    lambda x: x.fillna(x.mean()))

/var/folders/rl/0ykyj_7s1wx56ylgms0h9bxh0000gn/T/ipykernel_75710/3345242078.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  lambda x: x.fillna(x.mean()))
/var/folders/rl/0ykyj_7s1wx56ylgms0h9bxh0000gn/T/ipykernel_75710/3345242078.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_wide_cleaned[col] = df_wide_cleaned.groupby('Регистрационный номер')[col].transform(


In [22]:
for col in ['Коэффициент покрытия процентов по EBIT, %']:
    df_wide_cleaned[col] = df_wide_cleaned.groupby('Регистрационный номер')[col].transform(
    lambda x: x.fillna(x.mean()))

/var/folders/rl/0ykyj_7s1wx56ylgms0h9bxh0000gn/T/ipykernel_75710/2874757006.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  lambda x: x.fillna(x.mean()))
/var/folders/rl/0ykyj_7s1wx56ylgms0h9bxh0000gn/T/ipykernel_75710/2874757006.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_wide_cleaned[col] = df_wide_cleaned.groupby('Регистрационный номер')[col].transform(


In [23]:
for col in ['Коэффициент покрытия процентов по EBIT, %']:
    df_wide_cleaned[col] = df_wide_cleaned.groupby('Вид деятельности/отрасль')[col].transform(
    lambda x: x.fillna(x.mean()))

/var/folders/rl/0ykyj_7s1wx56ylgms0h9bxh0000gn/T/ipykernel_75710/3856988221.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_wide_cleaned[col] = df_wide_cleaned.groupby('Вид деятельности/отрасль')[col].transform(


такой подход может дать data leakedge


In [21]:
for column in df_wide_cleaned.loc[:, 'Выручка, RUB':'Соотношение совокупного долга к капиталу, %'].columns.to_list():
    low, high = df_wide_cleaned.loc[:, column].quantile([0.01, 0.99])
    df_wide_cleaned.loc[:, column] = df_wide_cleaned.loc[:, column].clip(lower=low, upper=high)

In [25]:
df_bancrupted = pd.read_csv('bankrupted.csv')

In [26]:
df_both = pd.concat([df_wide_cleaned, df_bancrupted])

In [34]:
def cut_history(df):
    out = []
    for firm, g in df.groupby("Регистрационный номер"):
        g = g.sort_values("Год")
        years = g["Год"].unique()
        max_len = len(years)
        if max_len < 3:
            continue
        L = np.random.randint(3, min(5, max_len) + 1)
        start_idx = np.random.randint(0, max_len - L + 1)
        window = years[start_idx : start_idx + L]
        out.append(g[g["Год"].isin(window)])
    return pd.concat(out)

df_cut = cut_history(df_wide_cleaned)

In [ ]:
df_cut.to_csv('../data/processed/healthy/healthy.csv', index=False)

In [ ]:
df_bancrupted = pd.read_csv('../data/processed/bancrupted/bankrupted_clean.csv')

In [41]:
df_both = pd.concat([df_cut, df_bancrupted])

In [42]:
df_both.drop(columns=['Наименование'], inplace=True)

In [ ]:
df_both.to_csv('../data/processed/healthy_bancrupted.csv', index=False)